# Week 3 — Feature Engineering & Data Prep: Quick Reference
**AI Engineer Lab · Phase 1**

Everything you need in one notebook. Copy-paste ready patterns for the most common preprocessing tasks.

---
**Sections:**
1. Missing Data — Profiling & Imputation
2. Categorical Encoding Decision Matrix
3. Feature Scaling Cheat Sheet
4. Feature Selection Toolkit
5. Dimensionality Reduction One-Liners
6. Production Pipeline Template
7. Data Leakage Prevention Checklist

In [ ]:
# ═══ STANDARD IMPORTS FOR ALL PREPROCESSING ═══
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler, Normalizer,
    PowerTransformer, QuantileTransformer,
    OrdinalEncoder, OneHotEncoder, LabelEncoder,
    FunctionTransformer
)
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, f_classif, f_regression,
    mutual_info_classif, mutual_info_regression,
    RFE, RFECV, SelectFromModel
)
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
import joblib
print('All imports ready.')

---
## 1. Missing Data — Profile & Impute

In [ ]:
# ── Profile missing data ──────────────────────────────────────────────
missing_report = pd.DataFrame({
    'count': df.isnull().sum(),
    'pct':   (df.isnull().mean() * 100).round(2)
}).query('count > 0').sort_values('pct', ascending=False)

# ── Simple imputation ─────────────────────────────────────────────────
num_imputer = SimpleImputer(strategy='median', add_indicator=True)    # numeric
cat_imputer = SimpleImputer(strategy='most_frequent')                  # categorical
ts_imputer  = SimpleImputer(strategy='mean')                           # time-series (or use ffill)

# ALWAYS fit on train only
X_train_imp = num_imputer.fit_transform(X_train[num_cols])
X_test_imp  = num_imputer.transform(X_test[num_cols])

# ── KNN imputation (correlated features) ─────────────────────────────
knn_imp = KNNImputer(n_neighbors=5)

# ── MICE — iterative, most accurate ──────────────────────────────────
mice_imp = IterativeImputer(max_iter=10, random_state=42)

---
## 2. Categorical Encoding

In [ ]:
# ── Ordinal (ordered categories) ─────────────────────────────────────
oe = OrdinalEncoder(
    categories=[['low', 'medium', 'high']],
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

# ── One-Hot (nominal, <15 categories) ────────────────────────────────
ohe = OneHotEncoder(
    drop='first',                   # avoid dummy variable trap
    sparse_output=False,
    handle_unknown='ignore'         # unseen categories → all zeros
)

# ── Frequency encoding (high cardinality, no leakage) ────────────────
freq_map = X_train['city'].value_counts().to_dict()
X_train['city_freq'] = X_train['city'].map(freq_map)
X_test['city_freq']  = X_test['city'].map(freq_map).fillna(0)

# ── Target encoding (high cardinality, use sklearn >=1.3) ────────────
# sklearn.preprocessing.TargetEncoder handles cross-fold automatically
from sklearn.preprocessing import TargetEncoder
te = TargetEncoder(target_type='continuous', smooth='auto')
X_train['city_te'] = te.fit_transform(X_train[['city']], y_train).flatten()
X_test['city_te']  = te.transform(X_test[['city']]).flatten()

---
## 3. Feature Scaling

In [ ]:
# ── StandardScaler — for Gaussian features, linear/SVM/NN ────────────
std_scaler = StandardScaler()            # mean=0, std=1

# ── MinMaxScaler — when bounded [0,1] needed ─────────────────────────
mm_scaler = MinMaxScaler()               # maps to [0, 1]

# ── RobustScaler — when outliers are present ─────────────────────────
rb_scaler = RobustScaler()               # uses median + IQR

# ── PowerTransformer — fix skewness before StandardScaler ────────────
pt = PowerTransformer(method='yeo-johnson')  # works for any real values

# ── Normalizer — row-wise unit norm (cosine similarity use cases) ─────
normalizer = Normalizer(norm='l2')       # each row has norm=1

# ── Always: fit on train, transform both ─────────────────────────────
X_train_sc = std_scaler.fit_transform(X_train)
X_test_sc  = std_scaler.transform(X_test)     # NOT fit_transform!

# ── Check skewness ────────────────────────────────────────────────────
# df.skew()  →  |value| > 1: moderate, > 2: severe → use PowerTransformer

---
## 4. Feature Selection

In [ ]:
# ── Step 1: Remove near-constant features ────────────────────────────
vt = VarianceThreshold(threshold=0.01)
X_vt = vt.fit_transform(X_train)

# ── Step 2: Remove highly correlated pairs ───────────────────────────
corr = pd.DataFrame(X_train).corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]

# ── Univariate selection (filter method) ─────────────────────────────
# Classification
sel_class = SelectKBest(mutual_info_classif, k=20)
# Regression
sel_reg   = SelectKBest(mutual_info_regression, k=20)

# ── RFECV — wrapper, finds optimal k automatically ───────────────────
from sklearn.linear_model import LogisticRegression
rfecv = RFECV(estimator=LogisticRegression(max_iter=1000), step=1, cv=5,
               scoring='roc_auc', min_features_to_select=5)
rfecv.fit(X_train_sc, y_train)
print(f'Optimal features: {rfecv.n_features_}')

# ── Tree importance (embedded) ────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
sfm = SelectFromModel(rf, threshold='mean', prefit=True)
X_important = sfm.transform(X_train)

---
## 5. Dimensionality Reduction

In [ ]:
# ── PCA (always scale first!) ─────────────────────────────────────────
X_sc = StandardScaler().fit_transform(X_train)

# Find k for 95% variance
pca_full = PCA()
pca_full.fit(X_sc)
k95 = np.argmax(np.cumsum(pca_full.explained_variance_ratio_) >= 0.95) + 1
print(f'Components for 95% variance: {k95}')

pca = PCA(n_components=0.95, random_state=42)  # auto-select k
X_pca = pca.fit_transform(X_sc)

# ── t-SNE (visualisation ONLY — never as model features) ─────────────
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_2d = tsne.fit_transform(X_sc[:5000])  # subsample for speed

# ── UMAP (faster, can use as features) ───────────────────────────────
# pip install umap-learn
# import umap
# reducer = umap.UMAP(n_components=2, n_neighbors=15, random_state=42)
# X_umap = reducer.fit_transform(X_sc)

# ── TruncatedSVD for sparse (text/TF-IDF) ────────────────────────────
svd = TruncatedSVD(n_components=100, random_state=42)
X_svd = svd.fit_transform(X_sparse)  # works on sparse matrices, no centering needed

---
## 6. Production Pipeline Template

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PRODUCTION PIPELINE TEMPLATE — Copy and adapt for any project
# ════════════════════════════════════════════════════════════════════════

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# 1. Identify column groups
num_features  = X.select_dtypes(include='number').columns.tolist()
cat_nominal   = ['colour', 'city']       # nominal categories
cat_ordinal   = ['size']                  # ordered categories

# 2. Build sub-pipelines per column type
numeric_pipe = Pipeline([
    ('imputer',   SimpleImputer(strategy='median', add_indicator=True)),
    ('transform', PowerTransformer(method='yeo-johnson')),   # fix skewness
    ('scaler',    StandardScaler()),
])

nominal_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
])

ordinal_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=[['small', 'medium', 'large', 'xlarge']],
        handle_unknown='use_encoded_value', unknown_value=-1
    )),
])

# 3. Combine with ColumnTransformer
preprocessor = ColumnTransformer([
    ('num',     numeric_pipe, num_features),
    ('nominal', nominal_pipe, cat_nominal),
    ('ordinal', ordinal_pipe, cat_ordinal),
], remainder='drop')

# 4. Full pipeline: preprocessor → optional PCA → model
from sklearn.ensemble import GradientBoostingClassifier
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    # ('pca', PCA(n_components=0.95)),   # uncomment if high-dimensional
    ('model', GradientBoostingClassifier(n_estimators=100, random_state=42)),
])

# 5. Cross-validate safely
cv_scores = cross_val_score(full_pipeline, X, y, cv=5, scoring='roc_auc')
print(f'CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# 6. Hyperparameter search over preprocessing + model
param_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'preprocessor__num__scaler': [StandardScaler(), RobustScaler()],
    'model__n_estimators': [100, 300],
    'model__max_depth': [3, 5, None],
}
gs = GridSearchCV(full_pipeline, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
gs.fit(X_train, y_train)
print(f'Best params: {gs.best_params_}')

# 7. Deploy
joblib.dump(gs.best_estimator_, 'model_pipeline.pkl')

# 8. Inference
pipe = joblib.load('model_pipeline.pkl')
predictions = pipe.predict(X_new_raw)   # raw data in, predictions out

---
## 7. Data Leakage Prevention Checklist

| Step | Rule | How to verify |
|------|------|---------------|
| Train/test split | **Split FIRST before any preprocessing** | Check that imputer.fit() is called after split |
| Imputation | `fit_transform(X_train)` then `transform(X_test)` | Imputer statistics should match training distribution |
| Scaling | `fit_transform(X_train)` then `transform(X_test)` | Scaler mean/std should be training data mean/std |
| Encoding | `fit_transform(X_train)` then `transform(X_test)` | Encoder vocabulary should come from training data |
| Target encoding | Use K-fold cross-encoding, not full-dataset means | Implement via K-fold or use `TargetEncoder` (sklearn ≥ 1.3) |
| Feature selection | `SelectKBest.fit()` on training fold only | Run inside `cross_val_score(pipeline, ...)` |
| PCA | `PCA.fit()` on training fold only | Run inside `cross_val_score(pipeline, ...)` |
| Cross-validation | `cross_val_score(pipeline, X, y)` not `cross_val_score(model, X_preprocessed, y)` | Preprocessing steps must be inside Pipeline |
| Deployment | One pkl file contains all fitted transformers | `pipe.predict(raw_data)` works without extra preprocessing |

---
## 8. Algorithm × Preprocessing Decision Table

| Algorithm | Needs Scaling? | Imputation | Encoding | Notes |
|-----------|---------------|------------|----------|-------|
| Linear Regression | YES (StandardScaler) | Required | OHE for nominal | PowerTransform skewed features |
| Logistic Regression | YES (StandardScaler) | Required | OHE for nominal | L1 for feature selection |
| SVM | YES (StandardScaler) | Required | OHE | Very sensitive to scale |
| KNN | YES (StandardScaler) | Required | OHE | Curse of dimensionality — select features |
| Decision Tree | NO | Required | Ordinal OK | Splits on thresholds; scale-invariant |
| Random Forest | NO | Required | Ordinal OK | Handles mixed types well |
| XGBoost / LightGBM | NO | Handles NaN | Ordinal OK | Can handle missing natively |
| Neural Network | YES (StandardScaler/MinMax) | Required | OHE or embedding | BatchNorm reduces sensitivity |
| K-Means | YES (StandardScaler) | Required | OHE | Very sensitive to scale and outliers |
| PCA | YES (StandardScaler) | Required | OHE | Must scale — variance-based |